In [1]:
import pandas as pd
import unicodedata
import re


In [2]:
url = r"C:\Users\clair\IronHack\Final Project\events-in-paris\Raw Data\que-faire-a-paris-.csv"
events = pd.read_csv(url, sep = ";")

In [3]:
# Remove columns that have more than 1000 null values

def drop_col_null_values(df):
    cols_to_drop = [col for col in df.columns if df[col].isnull().sum() > 1000]
    return df.drop(columns=cols_to_drop)


In [4]:
events = drop_col_null_values(events)

In [5]:
# Look at the remaining columns with highest number null values and decide to drop them or not
# title_event is a duplicate of title
# ID column is not necessary if we already have event_id
events.drop(columns=['ID','title_event', 'contact_organisation_name', "Crédit de l'image", 'group', 'locale', 'rank', 'weight', 'event_pets_allowed', "URL de l'image"], axis=1, inplace=True)


In [7]:
events.dropna(subset=['Coordonnées géographiques'], inplace=True)

In [8]:
events.drop(columns=['locations'], axis=1, inplace=True)

In [9]:
events = events.copy()
events = events[events['Date de fin'] > '2025-06-01']

In [10]:
split_themes = events['qfap_tags'].str.split(';', expand=True)
split_themes.columns = [f'theme{i+1}' for i in range(split_themes.shape[1])]
events = pd.concat([events, split_themes], axis=1)

In [11]:
events.drop('qfap_tags', axis=1, inplace=True)

In [ ]:
# creating latitude and longitude columns by splitting 'Coordonnées géographiques'
events[['latitude', 'longitude']] = events['Coordonnées géographiques'].str.split(',', expand=True)
events['longitude'] = pd.to_numeric(events['longitude'], errors='coerce')
events['latitude'] = pd.to_numeric(events['latitude'], errors='coerce')

In [13]:
events.drop('Coordonnées géographiques', axis=1, inplace=True)

In [14]:
import unicodedata

def clean_column_name(col):
    # Normalize accents and remove diacritics
    col = unicodedata.normalize('NFKD', col).encode('ascii', 'ignore').decode('utf-8')
    # Lowercase, replace apostrophes, remove punctuation, replace spaces
    col = (
        col.lower()
        .replace("'", "_")
        .replace("-", "_")
    )
    col = re.sub(r'[^\w\s]', '', col)      # remove punctuation
    col = re.sub(r'\s+', '_', col)         # replace whitespace with _
    return col

import re
events.columns = [clean_column_name(col) for col in events.columns]

In [16]:
# removing weird value
events = events[events['date_de_debut'] != '2530-02-27T19:00:00+01:00']

In [ ]:
# occurences is the column with all the event dates 
events['occurrences'].isnull().sum()

np.int64(491)

In [ ]:
# deleting events with no dates
events = events.copy()
events = events[events['occurrences'].notna()]

In [19]:
events['date_de_debut'].isnull().sum()

np.int64(0)

In [21]:
# Splitting occurences into different dates, one for each day of the event

# Split into new columns
new_cols = events['occurrences'].str.split(';', expand=True)

# Generate column names: date1, date2, ..., date150
col_names = [f'date{i+1}' for i in range(new_cols.shape[1])]

# Assign back to events with these names
events = events.copy()
events[col_names] = new_cols.copy()


In [22]:
events.columns

Index(['event_id', 'url', 'titre', 'chapeau', 'description', 'date_de_debut',
       'date_de_fin', 'occurrences', 'description_de_la_date', 'nom_du_lieu',
       ...
       'date2650', 'date2651', 'date2652', 'date2653', 'date2654', 'date2655',
       'date2656', 'date2657', 'date2658', 'date2659'],
      dtype='object', length=2682)

In [23]:
# List of the 2660 column names
date_cols = [f'date{i}' for i in range(1, 2660)]

# Apply the split to each column
for col in date_cols:
    events[col] = events[col].str.split('_').str[0]

In [24]:
events[date_cols] = events[date_cols].apply(pd.to_datetime, errors='coerce', utc=True)

In [25]:
events[date_cols] = events[date_cols].apply(lambda col: col.dt.strftime('%Y-%m-%d'))

In [26]:
# dropping redundant columns after creating the date columns
events.drop(columns=['occurrences', 'date_de_debut', 'date_de_fin', 'date_de_mise_a_jour'], axis=1, inplace=True)


In [28]:
# 1. Identify the 1660 date columns
date_cols = [col for col in events.columns if col.startswith('date')]

# 2. Melt the dataframe: convert wide format (many date columns) into long format (single column)
events_melted = events.melt(
    id_vars=[col for col in events.columns if col not in date_cols],
    value_vars=date_cols,
    var_name='date_number',
    value_name='date'
)

# 3. Drop rows where the date is missing (optional but often useful)
events_melted = events_melted.dropna(subset=['date'])

In [29]:
# cleaning description and description de la date columns
def remove_tags_and_replace(text):
    if not isinstance(text, str):
        return text
    # Replace each <...> tag with '-'
    return re.sub(r'<.*?>', ' ', text)


In [30]:
for col in ['description', 'description_de_la_date']:
    events_melted[col] = events_melted[col].apply(remove_tags_and_replace)


In [31]:
events_melted.to_csv('events.csv', index=False, encoding='utf-8')